# 05. Integrazione LLM e Telegram Bot 🤖🚕

Questo notebook documenta l'ultimo step del progetto: esporre il modello predittivo LightGBM (addestrato su 12 mesi di dati TLC) tramite un'interfaccia naturale usando modelli linguistici di grandi dimensioni (LLM).

> **Obiettivo**: Permettere agli utenti finali di chiedere la disponibilità dei taxi in linguaggio naturale (es. *"Quanti taxi ci sono a Midtown lunedì alle 8?"*) tramite un bot Telegram.

## Architettura del Sistema
Il modulo `llm_tool/` contiene un'architettura ibrida sviluppata su LangChain/LangGraph:

1. **Input Validator (`input_validator.py`)**: Utilizza Regex e un fallback con Ollama (temperature=0) per estrarre in modo deterministico la `{zona, ora, minuto, mese, giorno}` dal testo dell'utente.
2. **Taxi Predictor (`taxi_predictor.py`)**: Funziona come un tool callable che avvolge il modello LightGBM. Dato l'ID zona e il blocco orario, genera una previsione rapida e ricava le probabilità delle 5 classi (`Molto Difficile` → `Molto Facile`).
3. **Taxi Agent (`agent.py`)**: Usa un grafo ReAct (`langgraph.prebuilt.create_react_agent`) pilotato dal modello LLM (come `llama3.2:3b` o `gemma4:e2b`). L'LLM genera i parametri, chiama il tool logico, e restituisce una risposta testuale fluida nella lingua richiesta (Italiano o Inglese).
4. **Telegram Bot (`telegram_bot.py`)**: Interface di front-end integrata usando la libreria `python-telegram-bot`. Accetta sia comandi testuali liberi che invii rapidi via `/predict`.

## 1. Importazione Moduli
Carichiamo l'agente e il validatore dal nostro pacchetto.

In [ ]:
from llm_tool import get_agent, get_validator

agent = get_agent()
validator = get_validator()

## 2. Test Input Validator (Estrazione Deterministica Regex/LLM)
L'Input Validator cerca per prima cosa regole Regex per maggiore velocità. Se manca qualcosa, sfrutta prompt di estrazione via JSON usando ChatOllama.

In [ ]:
queries = [
    "Quanti taxi ci sono a JFK di domenica mattina ad agosto?",
    "Taxi a Midtown lunedì alle 8",
]

for q in queries:
    print(f"User: '{q}'")
    params = validator.extract(q)
    resolved = validator.validate_and_resolve(params)
    print(f"  -> Params estratti: {params}")
    print(f"  -> Location ID: {resolved.get('location_id')}\n")

## 3. Chiamata Rapida: `direct_predict`
Questa è l'esecuzione usata dal comando `/predict` di Telegram. Bypassa la conversazione testuale LLM per interfacciarsi direttamente col predictor e generare un template stringa formattato, massimizzando la velocità.

In [ ]:
import sys

# Parametri: zona, mese, giorno_settimana (0=Lun), ora
resp = agent.direct_predict("Times Square", 7, 5, 22) # Sabato sera a Luglio
print(resp)

## 4. Agente Conversazionale LangGraph
Questa è l'esecuzione completa. Si affida al `ChatOllama` con `llama3.2:3b` per comprendere, validare i parametri o porre domande di disembiguazione se incompleti e poi generare un resoconto finale.

*(Nota: Assicurati di avere Ollama attivo ed in esecuzione).* 

In [ ]:
try:
    answer = agent.chat("Ciao! E' facile trovare un taxi a Brooklyn venerdì alle 17 a gennaio?")
    print("Risposta LLM:\n")
    print(answer)
except Exception as e:
    print(f"Errore durante la generazione dell'LLM (controlla che il servizio locale Ollama sia avviato): {e}")

## 5. Avvio Bot Telegram (Stand-alone)
Per avviare il tool a disposizione del pubblico, basta configurare il file `.env` aggiungendo il proprio token API rilasciato da BotFather:
```
TELEGRAM_TOKEN=12345678:AABBCcDDEefG_hil...
```
Dopodichè, è possibile eseguire lo script direttamente da terminale o task di background:

```bash
python -m llm_tool.telegram_bot
```
Questo comando inizializzerà globalmente il predictor e incomincerà ad accettare le richieste in polling.